# LLM architecture, tokenization and quantization

In this exercise, I explore the structure of typical LLM based transformers and the use of tokenizers. The goal is to provide an introduction to LLMs by accessing directly its archtecture.

**Transormer and HuggingFace**

We use the transformer library from Huggingface to explore LLM based models (also called pre-traine model) and their architectures. We alos experiment with instruct models and produce some inference and compare outputs. I use two llm models:

- Llama3 8B from Meta
- phi 4 from Microsoft

LLama 3 is an earlier version of Llama which is often used in many applications while Phi4 from Microsoft is also known for its small size a good performance.

**Quantization**

LLMs are usually are typically large models that require a large compute resource along with GPUs. To reduce the size of the model in memory LLMs are often reduced in size using quantization.
With LLama 3 8B, we will use quantization for the instruct model to reduce its size in memory using the 'bitsandbytes' library.

**LLM architecture**

In essence, LLMs are built using a stack of transformers. There four main components in a LLM:

- tokenizer: converting text to int number
- embedding: mapping text to dense vector space
- transformer layers with attention: enriching the embedding with context.
- classifyer layer: mlp layer carrying out the classifcation task (predicting the next token).

## Interesting links:

- huggingFace:
https://medium.com/@birendrasharma0226/key-differences-among-model-classes-a060ebea34c6

- quantization and qlora:

https://mccormickml.com/2024/09/14/qlora-and-4bit-quantization/

- instructgpt paper:

https://www.youtube.com/watch?v=RkFS6-GwCxE&t=8s

- phi 4 paper:
https://arxiv.org/pdf/2412.08905

- llama 3 paper:

# Set up environment and load libraries

- load libraries
- install packages and tools
- authenticate to google drive and gcp account

If using an virtual env we can use a requirements or lockfile. Here is a list of required packages:

```
transformers==4.50.0
accelerate==1.5.2
datasets==3.4.1
trl==0.16.0
python-dotenv==1.0.0
peft==0.15.0
wandb==0.19.8

```

In [1]:
!pip install openai
#!pip install trl==0.16
!pip install trl
!pip install bitsandbytes
!pip install objsize
#!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.0 MB/s eta 0:00:00


In [2]:
!pip install pynvml #to monitor GPU usage

In [3]:
import gc
import os
import torch
from google.colab import userdata
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TextStreamer,
    pipeline,
)

General purpose libraries and packages used in this exercise.

In [4]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib import colors
import matplotlib.patches as mpatches
import seaborn as sns

import numpy as np
import subprocess
import pandas as pd
import os, glob
import zipfile

from pathlib import Path

sns.set_style('darkgrid')
#pd.set_option('display.max_colwidth', None)
!apt install unzip
import urllib
import re
import math
from datetime import datetime
from copy import deepcopy
from numpy.core.multiarray import datetime_as_string
import os
import numpy.ma as ma

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unzip is already the newest version (6.0-26ubuntu3.2).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


/tmp/ipykernel_8916/975698043.py:25: DeprecationWarning: numpy.core.multiarray is deprecated and has been renamed to numpy._core.multiarray. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.multiarray.datetime_as_string.
  from numpy.core.multiarray import datetime_as_string


In [5]:
#Used in defining functions
from typing import List, Tuple, Dict, Any
from pandas.core.arrays import boolean
import pandas as pd

In [6]:
import openai

In [7]:
!sudo apt install tree
!pip install tree
!pip install python-dotenv

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 0s (163 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package tree.
(Reading database ... 121852 files and directories currently install

In [8]:
#GCP account authentification
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

Authenticated


In [9]:
from google.colab import drive

drive.mount('/content/gdrive')

Mounted at /content/gdrive


# Environment configuration

Let's upload .env and set up the other environmental variables using secrets. To run this notebook we need the following environment variables:

- hf_token: hugging face access token


In [10]:
from google.colab import files
uploaded = files.upload()

Saving .env to .env


In [11]:
import dotenv
dotenv.load_dotenv('./.env')

from google.colab import userdata
#wb_token = userdata.get('WANDB_API_KEY')
hf_token = userdata.get('HF_TOKEN')
from huggingface_hub import login

# Replace 'YOUR_HF_TOKEN' with your actual token
#your_token = "hf_YOUR_HF_TOKEN_HERE"
login(token=hf_token,add_to_git_credential=True)

# Functions
In the next part of the script, we declare all the functions used in the sripts.  It is good practice to place functions at the beginning of a script or in an external source file. Here are the 13 functions used:

In [12]:
def create_dir_and_check_existence(path: str) -> str:
  '''
  Create a new directory

  :param path: path to the directory
  :return: path to the directory
  '''

  try:
    os.makedirs(path)
  except:
    print ("directory already exists")
  return path



# Parameters and Arguments

It is good practice to set all parameters and input arguments at the beginning of the script. This allows for better control and can make modifications of the scripts for other applications easier. Some arguments relate to path directories, input files and general parameters for use in the analyses (e.g. proportion of hold out).


In [13]:
############################################################################
#####  Parameters and argument set up ###########

#ARG 1
in_dir = '/content/gdrive/MyDrive/Colab Notebooks/Large-Language-Models-applications/llm-architecture-and-tokenization/data/'
#ARG 2
out_dir = "/content/gdrive/MyDrive/Colab Notebooks/Large-Language-Models-applications/llm-architecture-and-tokenization"
#ARGS 3:
create_out_dir=True #create a new ouput dir if TRUE
#ARG 4
out_suffix = "20260105" #output suffix for the files and ouptut folder
#ARG 6
random_seed=105 # set seed for reproducibility of results

# Model
base_model = "meta-llama/Meta-Llama-3-8B"
base_model_insruct = "meta-llama/Meta-Llama-3-8B-Instruct"
PHI = "microsoft/Phi-4-mini-instruct"

In [14]:
#set up the working directory
#Create output directory

if create_out_dir==True:
    out_dir_new = "output_data_"+out_suffix
    out_dir = os.path.join(out_dir,"outputs",out_dir_new)
    create_dir_and_check_existence(out_dir)
    os.chdir(out_dir)        #set working directory
else:
    os.chdir(out_dir) #use working dir defined earlier


directory already exists


In [15]:
print(out_dir)

/content/gdrive/MyDrive/Colab Notebooks/Large-Language-Models-applications/llm-architecture-and-tokenization/outputs/output_data_20260105


First let's check if we are connected to a GPU. We are using T4 here.


In [16]:
# Check Google Colab GPU
# this is from Ed Donner https://github.com/ed-donner/llm_engineering/tree/main

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Success - Connected to a T4")
  else:
    print("NOT CONNECTED TO A T4")

Mon Mar  9 03:03:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   39C    P8             16W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# 1.Accessing LLMs with HuggingFace: Inference


Let's now try to chat with one of the model in HuggingFace. We can use HuggingFace pipeline with the task set to "text_generation" and pointing to the phi-4 or LLama 3 models. Note that when using the pipeline functionality, we do not need to provide a tokenizer or load separaterly a model. We can just reference the model name.

Generation using this approach may be slow. We will revisit the size the speed of inference later on.

## 1.1 Accessing LLama 3 with HuggingFace pipeline

In [17]:
import torch
from transformers import pipeline

#chose a specific model:
model_name =  "meta-llama/Llama-3.2-3B-Instruct"
base_model_insruct = "meta-llama/Meta-Llama-3-8B-Instruct"

Now let's use the pipeline functionaly:
- task: 'text generation
- model: name of the model used
- device: set to auto, it can detect if there is GPU

In [18]:
pipe = pipeline(
    "text-generation",
    model=base_model_insruct,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

We can pass the instructions using the general structure we use with the open AI package. In fact, the structure below is common to many LLM models (e.g. ChatGPT, Gemini, Anthropic Claude etc.). As a reminder, we structure the inputs as a list of dictionaries. For instance:
- user dictionary: with role set to user and the content being the task instruciton.
- system dictionary: with the role set to system and the content being the instruction that the LLM must follow (e.g. setting tone, persona, audience etc.).
- assistant dictionionary: with the role set to assistant and content the response from the LLM. This is used in few shot learning.


In the response below the messages list only containts the first two entries.

In [19]:
user_message = "How do clouds form?"
system_prompt = "You are an expert Meteorologist who can explain concept to lay people"
messages = [
    {"role": "user", "content": user_message},
    {"role": "system", "content": system_prompt }
]

This steps takes 6 minutes

In [20]:
# 3. Generate the text
llm_outputs = pipe(
    messages,
    max_new_tokens=500,  #max output tokens numbers for the repsonse
    do_sample=False,      # Enable sampling (recommended for creative tasks
    eos_token_id=pipe.tokenizer.eos_token_id
)

# 4. Print the generated response
# The output is a list of dictionaries; access the generated text from the first item
print(llm_outputs[0]['generated_text'][-1]['content'])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Both `max_new_tokens` (=500) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Clouds! Those majestic, ever-changing wonders of the sky. Clouds are a crucial part of our planet's weather and climate system, and understanding how they form is essential for predicting the weather and appreciating the beauty of the atmosphere.

So, let's dive into the fascinating world of cloud formation!

**The Basic Ingredients**

Clouds form when a combination of three essential ingredients come together:

1. **Water Vapor**: Water vapor is the gaseous state of water that's present in the air. It's released into the atmosphere through various processes, such as evaporation from oceans, lakes, and rivers, as well as transpiration from plants.
2. **Cool Air**: Cool air is necessary for water vapor to condense into tiny droplets. This can happen when air rises over a mountain, cools as it expands, or when it's cooled by contact with a cold surface.
3. **Nuclei**: Nuclei are tiny particles in the air that provide a surface for water vapor to condense onto. These particles can be natu

## 1.2 Accessing Phi-4 with HuggingFace pipeline

As an example we will also use Phi-4-mini-instruct from Microsoft:

https://huggingface.co/microsoft/Phi-4-mini-instruct

This takes 11 minutes.


In [21]:
pipe2 = pipeline(
    "text-generation",
    model="microsoft/Phi-4-mini-instruct",
    model_kwargs={"torch_dtype": "auto"},
    device_map="auto",
)

message = "How do clouds form?"
system_prompt = "You are an expert Meteorologist who can explain concept to lay people"
messages = [
    {"role": "user", "content": message},
    {"role": "system", "content": system_prompt }
]

llm_outputs = pipe2(messages,
                   temperature=0.7,     # text randomness
                   top_p=0.9,
                   do_sample=True,      # Enable sampling (recommended for creative tasks
                   eos_token_id=pipe.tokenizer.eos_token_id,
                   max_new_tokens=500)
#print(llm_outputs[0]["generated_text"][-1])
print(llm_outputs[0]['generated_text'][-1]['content'])

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'top_p', 'max_new_tokens', 'do_sample', 'temperature', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Clouds form through a process that involves the cooling of air to its dew point, where the air becomes saturated with water vapor, and the vapor condenses into tiny water droplets or ice crystals, depending on the temperature. This process can be broken down into several steps:


1. **Evaporation**: Water from oceans, lakes, rivers, or moist soil turns into water vapor due to the heat from the sun.

2. **Rising Warm Air**: This warm, moist air rises because it is less dense than the surrounding cooler air.

3. **Cooling**: As the air rises, it expands and cools due to the decrease in atmospheric pressure at higher altitudes.

4. **Condensation**: When the air cools to its dew point, the water vapor condenses onto tiny particles in the air, such as dust, pollen, or sea salt, forming water droplets or ice crystals.

5. **Cloud Formation**: These droplets or crystals cluster together to become visible as a cloud. Depending on the conditions, they may remain suspended in the atmosphere or 

#2.Explore tokenizers for LLM instructs

Before loading LLM models it is useful to look at the tokenizers used. Each LLM model series has its own tokenizer. Note these tokenizers usually use a subword/piece approach which means that a token may correspond to a word or a subword. For instance, the word 'playing' may be tokenized in the following way:

"play" + "ing"

- play: integer value 1003
- ing: integer value 7772

This approach reduces the number of token used (think of players, plays, playing) and allows the LLM to learn useful endings and lemma/stems in the language(s) used for training. In the case of 'playing' the lemma/stem would be the subword 'play'.

##2.1. LLama 3 8B tokenizer

We use the LLama 3 8B model from Meta accessed through the hugging face hub:

- https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct

We first take a look at the tokenizer by passing a short sentence and examining the integer values associated with each token. LLama 3 uses a Byte Pair Encoding tokenizer. The vocabulary size of token is 128,256 tokens.


Links:

- LLama3 tokenizer:





In [22]:
#tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3-8B', trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3-8B-Instruct', trust_remote_code=True)
print(type(tokenizer))
print(type(tokenizer.vocab))
print(len(tokenizer.vocab))

#meta-llama/Meta-Llama-3-8B-Instruct

<class 'transformers.tokenization_utils_tokenizers.TokenizersBackend'>
<class 'dict'>
128256


In [23]:
text = "I am a data scientist learning about LLMs and generative AI."
tokens = tokenizer.encode(text)
print(f"Token IDs: {tokens}")
print(f"Token string subwords {tokenizer.batch_decode(tokens)}")

Token IDs: [128000, 40, 1097, 264, 828, 28568, 6975, 922, 445, 11237, 82, 323, 1803, 1413, 15592, 13]
Token string subwords ['<|begin_of_text|>I am a data scientist learning about LLMs and generative AI.']


In [24]:
subwords = [tokenizer.decode([t]) for t in tokens]
print(f"Subwords: {subwords}")

Subwords: ['<|begin_of_text|>', 'I', ' am', ' a', ' data', ' scientist', ' learning', ' about', ' L', 'LM', 's', ' and', ' gener', 'ative', ' AI', '.']


Note that as we mentioned earlier, tokens are note equivalent to words or characters. Tokens correspond to subword defined using Byte-Pair-Encoding (BPE).

In [25]:
print(f"Number of tokens: {len(tokens)}")
print(f"Number of words: {len(text.split(' '))}") #number of words
print(f"Number of Characters: {len(text)}")

Number of tokens: 16
Number of words: 11
Number of Characters: 60


We can also use the tokenizer to return a pyTorch tensor with input attention mask.

In [26]:
text = "I am a data scientist learning about LLMs and generative AI."
input = tokenizer(text, return_tensors="pt")
print(input)

{'input_ids': tensor([[128000,     40,   1097,    264,    828,  28568,   6975,    922,    445,
          11237,     82,    323,   1803,   1413,  15592,     13]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


There are a few special tokens:

In [27]:
print(tokenizer.eos_token)

special_tokens_dict = {}
list_special_tokens_df = []
for special_token_name, special_token_string in tokenizer.special_tokens_map.items():
    # The convert_tokens_to_ids method takes a token string and returns its ID
    special_token_id = tokenizer.convert_tokens_to_ids(special_token_string)
    df = pd.DataFrame({"token_name":[special_token_name],
                  "token_string":[special_token_string],
                  "token_id":[special_token_id]})
    list_special_tokens_df.append(df)

df_special_tokens = pd.concat(list_special_tokens_df).reset_index()
df_special_tokens


<|eot_id|>


,index,token_name,token_string,token_id
0,0,bos_token,<|begin_of_text|>,128000
1,0,eos_token,<|eot_id|>,128009


We can explore all the reserved token_ID used by the LLama model. There are in total 256 reserved token with some already assigned.

In [28]:
tokenizer.get_added_vocab()

{'<|begin_of_text|>': 128000,
 '<|end_of_text|>': 128001,
 '<|reserved_special_token_0|>': 128002,
 '<|reserved_special_token_1|>': 128003,
 '<|reserved_special_token_2|>': 128004,
 '<|reserved_special_token_3|>': 128005,
 '<|start_header_id|>': 128006,
 '<|end_header_id|>': 128007,
 '<|reserved_special_token_4|>': 128008,
 '<|eot_id|>': 128009,
 '<|reserved_special_token_5|>': 128010,
 '<|reserved_special_token_6|>': 128011,
 '<|reserved_special_token_7|>': 128012,
 '<|reserved_special_token_8|>': 128013,
 '<|reserved_special_token_9|>': 128014,
 '<|reserved_special_token_10|>': 128015,
 '<|reserved_special_token_11|>': 128016,
 '<|reserved_special_token_12|>': 128017,
 '<|reserved_special_token_13|>': 128018,
 '<|reserved_special_token_14|>': 128019,
 '<|reserved_special_token_15|>': 128020,
 '<|reserved_special_token_16|>': 128021,
 '<|reserved_special_token_17|>': 128022,
 '<|reserved_special_token_18|>': 128023,
 '<|reserved_special_token_19|>': 128024,
 '<|reserved_special_token_

##2.2 Phi4 tokenizer

Phi4 tokenizer is available from HuggingFace:

- https://huggingface.co/microsoft/Phi-4-mini-instruct


In [29]:
tokenizer_phi4 = AutoTokenizer.from_pretrained("microsoft/Phi-4-mini-instruct")
print(type(tokenizer_phi4.vocab))
print(len(tokenizer_phi4.vocab))

<class 'dict'>
200029


In [30]:
text = "I am a data scientist learning about LLMs and generative AI"
tokens = tokenizer_phi4.encode(text)
print(tokenizer_phi4)
print(tokenizer_phi4.batch_decode(tokens))

TokenizersBackend(name_or_path='microsoft/Phi-4-mini-instruct', vocab_size=200019, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	199999: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	200018: AddedToken("<|endofprompt|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	200019: AddedToken("<|assistant|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=True),
	200020: AddedToken("<|end|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=True),
	200021: AddedToken("<|user|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=True),
	200022: AddedToken("<|system|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=True),

In [31]:
print(f"Number of tokens: {len(tokens)}")
print(f"Number of words: {len(text.split(' '))}") #number of words
print(f"Number of Characters: {len(text)}")

Number of tokens: 14
Number of words: 11
Number of Characters: 59


In [32]:
tokenizer_phi4.special_tokens_map

{'bos_token': '<|endoftext|>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<|endoftext|>',
 'pad_token': '<|endoftext|>'}

In [33]:
print(tokenizer_phi4.eos_token)
del df
list_special_tokens_df = []
for special_token_name, special_token_string in tokenizer_phi4.special_tokens_map.items():
    # The convert_tokens_to_ids method takes a token string and returns its ID
    special_token_id = tokenizer_phi4.convert_tokens_to_ids(special_token_string)
    df = pd.DataFrame({"token_name":[special_token_name],
                  "token_string":[special_token_string],
                  "token_id":[special_token_id]})
    list_special_tokens_df.append(df)

df_special_tokens = pd.concat(list_special_tokens_df).reset_index()
df_special_tokens

<|endoftext|>


,index,token_name,token_string,token_id
0,0,bos_token,<|endoftext|>,199999
1,0,eos_token,<|endoftext|>,199999
2,0,unk_token,<|endoftext|>,199999
3,0,pad_token,<|endoftext|>,199999


In the case of Phi-4, there are only 12 added special tokens.

In [34]:
tokenizer_phi4.get_added_vocab()

{'<|endoftext|>': 199999,
 '<|endofprompt|>': 200018,
 '<|assistant|>': 200019,
 '<|end|>': 200020,
 '<|user|>': 200021,
 '<|system|>': 200022,
 '<|tool|>': 200023,
 '<|/tool|>': 200024,
 '<|tool_call|>': 200025,
 '<|/tool_call|>': 200026,
 '<|tool_response|>': 200027,
 '<|tag|>': 200028}

##2.3 Chat template from huggingface

In [35]:
message = "How do clouds form?"
system_prompt = "You are an expert Meteorologist who can explain concept to lay people"
messages = [
    {"role": "user", "content": message},
    {"role": "system", "content": system_prompt }
]


In [36]:
print("Llama 3 8B:")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nPhi 4:")
print(tokenizer_phi4.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

Llama 3 8B:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

How do clouds form?<|eot_id|><|start_header_id|>system<|end_header_id|>

You are an expert Meteorologist who can explain concept to lay people<|eot_id|><|start_header_id|>assistant<|end_header_id|>



Phi 4:
<|user|>How do clouds form?<|end|><|system|>You are an expert Meteorologist who can explain concept to lay people<|end|><|assistant|>


#3.Exploring LLM model in memory without quantization

##3.1 Loading model into memory for direct inference

Using code from hugging face:
- https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct

We import the "AutoModelForCausalLM" because LLMs are generative models that are autoregressive. They predict one word at a time. In the code below, we load in memory LLama 3 8B without reducing its size.

Links:
- https://moocaholic.medium.com/fp64-fp32-fp16-bfloat16-tf32-and-other-members-of-the-zoo-a1ca7897d407
- https://en.wikipedia.org/wiki/Bfloat16_floating-point_format




Before running the cell, we need to restart the session because all the GPU memory is  used up.

In [17]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
)

messages = [
    {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
    {"role": "user", "content": "Who are you?"},
]

# Get the BatchEncoding object containing input_ids and attention_mask
encoded_inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

# Extract the input_ids tensor and attention_mask tensor
input_ids_tensor = encoded_inputs['input_ids']
attention_mask_tensor = encoded_inputs['attention_mask']

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(
    input_ids_tensor, # Pass the actual input_ids tensor
    attention_mask=attention_mask_tensor, # Pass the attention_mask
    max_new_tokens=256,
    eos_token_id=terminators,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
)
response = outputs[0][input_ids_tensor.shape[-1]:] # Use the shape of the actual input_ids tensor
print(tokenizer.decode(response, skip_special_tokens=True))

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Arrrr, me hearty! Me name be Captain Chat, the scurviest pirate chatbot to ever sail the seven seas... er, chat the digital waves! Me be here to swab yer digital decks, answer yer questions, and share me treasure o' knowledge with ye! So hoist the sails and set course fer a swashbucklin' good time, matey!


This is using 15 giga of memory! We can obtain the size in memory using the method "get_memory_footprint()
as well as the number of parameters using "num_parameters".

In [18]:

print(f"Number of parameters: {model.num_parameters()}") #this is roughly 8 B
model.get_memory_footprint()/(1024 * 1024 *1024)

Number of parameters: 8030261248


14.95752763748169

We can also track the GPU usage using nvidia-smi.

In [19]:
!nvidia-smi

Mon Mar  9 02:59:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   53C    P0             71W /   72W |   15598MiB /  23034MiB |     97%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Let's compute the size in memory use an approximation:

- brain float 16 bits: uses 16 bits per parameter to store real number values
- 8 billions parameters for Llama 3
- convert the bits to bytes by dividing by 8
- convert to Gb by dividing by 1 billion or more precisely 1024*12024*1024 because there are 1 billion bytes in a gb.

```
8e8 * (16/8) / (1024*1024*1024)
```

Note that this is a rough approximation because it does not take into account other memory usage that are necessary during inference such as the context size and architecture.

In [20]:
8e9*(16/8)/(1024*1024*1024)

14.901161193847656

##3.2.Exploring Llama3 model architecture

Llama 3 was created by Meta in April 2024. It is an open model that was described in a paper. Weights were released and several iterations and updated models were relesased later on. Llama 3 comes in different flavors:

-8B: small model with 8 billions parameters
-70B: medium model with 70 billions parameters
-405B: large model with 405 billions parameters

We use the LLalma 3 with a context length of 8,192k and a vocabulary size of 128k. The use of RoPE allows for large effective context up to 32k (verify this). Note that LLama 3 models were update in July 2024 to version 3.1. While the parameters size remain the same, they can be multilingual and have larger context size.


Links:

- https://arxiv.org/pdf/2407.21783

In [21]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  

- wte: word token embedding
- wpe: word position embedding
- h : transformer decoder layer
- ln_f: final normalization layer
- lm_head: mlp layer classification layer with vocabulary size output (50257) used to predict the next word token.

**Transfomer decoder***:

- ln_1: layer normalization 1
- attn: multi head attention layer
- mlp: mulitlayer percepteron

In [22]:
model.model.layers

ModuleList(
  (0-31): 32 x LlamaDecoderLayer(
    (self_attn): LlamaAttention(
      (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
      (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
      (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
      (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
    )
    (mlp): LlamaMLP(
      (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
      (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
      (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
      (act_fn): SiLUActivation()
    )
    (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
    (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
  )
)

In [23]:
print(type(model.model.layers))
print(type(model.model.layers[0]))
model.model.layers[0]

<class 'torch.nn.modules.container.ModuleList'>
<class 'transformers.models.llama.modeling_llama.LlamaDecoderLayer'>


LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
    (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
    (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
    (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
    (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
)

In [24]:
model.model.layers[0].self_attn

LlamaAttention(
  (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
  (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
  (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
  (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
)

Let's now take a look at the weights from the LLama 3 model. HuggingFace stores this model using pyTorch so we will access it the same way we do for any pyTorch model:

In [25]:
# 1. See all layer names (the model architecture)
print("Model Layer Names:")
for name, param in model.named_parameters():
    print(f"{name} | Shape: {param.shape}")

# 2. View specific weight values (e.g., from the first layer's self-attention)
# 'model.layers.0.self_attn.q_proj.weight' is a common Llama 3 layer name
specific_weight = model.state_dict()['model.layers.0.self_attn.q_proj.weight']

print("\nSample Weight Values from first layer (Query Projection):")
print(specific_weight[0:5, 0:5]) # Prints a 5x5 slice of the weight matrix

Model Layer Names:
model.embed_tokens.weight | Shape: torch.Size([128256, 4096])
model.layers.0.self_attn.q_proj.weight | Shape: torch.Size([4096, 4096])
model.layers.0.self_attn.k_proj.weight | Shape: torch.Size([1024, 4096])
model.layers.0.self_attn.v_proj.weight | Shape: torch.Size([1024, 4096])
model.layers.0.self_attn.o_proj.weight | Shape: torch.Size([4096, 4096])
model.layers.0.mlp.gate_proj.weight | Shape: torch.Size([14336, 4096])
model.layers.0.mlp.up_proj.weight | Shape: torch.Size([14336, 4096])
model.layers.0.mlp.down_proj.weight | Shape: torch.Size([4096, 14336])
model.layers.0.input_layernorm.weight | Shape: torch.Size([4096])
model.layers.0.post_attention_layernorm.weight | Shape: torch.Size([4096])
model.layers.1.self_attn.q_proj.weight | Shape: torch.Size([4096, 4096])
model.layers.1.self_attn.k_proj.weight | Shape: torch.Size([1024, 4096])
model.layers.1.self_attn.v_proj.weight | Shape: torch.Size([1024, 4096])
model.layers.1.self_attn.o_proj.weight | Shape: torch.Si

Let's clean up the memory. We will use code modified from from google gemini ai to remove the model from memory.

In [26]:
import torch
import gc

# Assuming 'llm_model' is the variable holding your loaded model
if 'model' in locals():
    # Optional: Move the model to CPU first to ensure all GPU references are handled
    # Note: For very large models, this might require significant CPU RAM
    # llm_model.to("cpu")

    # Explicitly delete the model variable
    del model

# Delete any other variables that might reference GPU tensors (e.g., inputs, outputs)
# del inputs
# del outputs

# Run Python's garbage collector
gc.collect()

# Empty the PyTorch CUDA cache
torch.cuda.empty_cache()


In [27]:
!nvidia-smi

Mon Mar  9 03:00:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   53C    P0             30W /   72W |   15580MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

#4.Using model with quantization: lower precision to load models

Before running this section, we need to restart the sesssion to clear the memory.

## 4.1 loading model

Let's load in the model but using BitsAndBytes to reduce the size in memory.

To load the model, we must now provide a configuration dictionary:

- load_in_4bit: if true use 4 bit when loading model

- bnb_4bit_use_double_quant: if true, this will quantitize both weights and constants.

- bnb_4bit_compute_dtype: this sets the data type and precision used during training and inference. For instance if using bfloat16, all computation will be using that format during inference and training. Note that during storage the model uses 4bit format.

- bnb_4bit_quant_type: format/data type to use for quantization. Most commonly "normalize 4 bit" or "nf4" for LLMs.


In the current example, We use normalize 4 bits float (nf4) and bfloat16 for inference.

links:
- https://dbrpl.medium.com/loading-an-llm-in-4-bits-using-bitsandbytes-5c2f86e57692

- https://www.maartengrootendorst.com/blog/quantization/


In [17]:
#Note: need to restart the session to clear the memory before running this.

torch_dtype = torch.bfloat16

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_quant_type="nf4"
)

# The model

model = AutoModelForCausalLM.from_pretrained(base_model_insruct,
                                             device_map="auto",
                                             quantization_config=quant_config)

print(model.num_parameters())
print(type(model))


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

8030261248
<class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>


In [18]:
!ls -ltra ~/.cache/huggingface/hub

!du -sh '~/.cache/huggingface/hub/models--meta-llama--Meta-Llama-3-8B-Instruct'

total 16
drwxr-xr-x 3 root root 4096 Mar  9 03:04 .locks
drwxr-xr-x 4 root root 4096 Mar  9 03:04 .
drwxr-xr-x 4 root root 4096 Mar  9 03:04 ..
drwxr-xr-x 6 root root 4096 Mar  9 03:05 models--meta-llama--Meta-Llama-3-8B-Instruct
du: cannot access '~/.cache/huggingface/hub/models--meta-llama--Meta-Llama-3-8B-Instruct': No such file or directory


Let's estimate the memory size using for this quantized model:

In [19]:
model.get_memory_footprint()/(1024 * 1024 * 1024)

5.2075276374816895

We can also note that the model shows that the different layers use 4bit.

In [20]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRM

We also note that the attention layers use a form of Linear4bit in the Q,K and V inputs.

In [21]:
model.model.layers[0].self_attn

LlamaAttention(
  (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
  (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
  (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
  (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
)

In [22]:
model.model.layers[0].self_attn.q_proj

Linear4bit(in_features=4096, out_features=4096, bias=False)

Using quantization, we reduce the model size from 15 giga to 5.2 giga bites.

##4.2 Model inference: generating text with instruct model

In [23]:
base_model_insruct

'meta-llama/Meta-Llama-3-8B-Instruct'

In [24]:
message = "How do clouds form?"
system_prompt = "You are an expert Meteorologist who can explain concept to lay people"
messages = [
    {"role": "user", "content": message},
    {"role": "system", "content": system_prompt }
]
messages


[{'role': 'user', 'content': 'How do clouds form?'},
 {'role': 'system',
  'content': 'You are an expert Meteorologist who can explain concept to lay people'}]

In [25]:
tokenizer = AutoTokenizer.from_pretrained(base_model_insruct)
tokenizer.pad_token = tokenizer.eos_token

# The apply_chat_template returns a BatchEncoding object
encoded_inputs = tokenizer.apply_chat_template(messages,
                                               return_tensors="pt").to("cuda")

# Extract input_ids and attention_mask from the BatchEncoding object
input_ids = encoded_inputs['input_ids']
attention_mask = encoded_inputs['attention_mask']

outputs = model.generate(input_ids=input_ids,
                         attention_mask=attention_mask,
                         max_new_tokens=80)
outputs[0]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


tensor([128000, 128006,    882, 128007,    271,   4438,    656,  30614,   1376,
            30, 128009, 128006,   9125, 128007,    271,   2675,    527,    459,
          6335,  40732,  16549,    889,    649,  10552,   7434,    311,  11203,
          1274, 128009, 128006,  78191, 128007,    271,  16440,     82,    527,
           264,  27387,  25885,    430,   1514,    264,  16996,   3560,    304,
          1057,  11841,    596,   9282,    323,  10182,     13,   2100,     11,
          1095,    757,   1464,   1523,    279,   1920,    315,   1268,  30614,
          1376,    304,    264,   1648,    430,    596,   4228,    311,   3619,
           382,    334,   8468,    220,     16,     25,  10641,  96649,   1035,
         16440,     82,   1212,  30164,    994,    279,   7160,  77662,    279,
          9420,    596,   7479,     11,  14718,   3090,    311,  60150,    349,
           505,  54280,     11,  44236,     11,  36617,     11,    323,   1524,
         55682], device='cuda:0')

Above the outputs is a pyTorch tensor with ID for each token.
We can see the input tensor in pyTorch loaded on the Cuda (GPU) device:

In [34]:
encoded_inputs

{'input_ids': tensor([[128000, 128006,    882, 128007,    271,   4438,    656,  30614,   1376,
             30, 128009, 128006,   9125, 128007,    271,   2675,    527,    459,
           6335,  40732,  16549,    889,    649,  10552,   7434,    311,  11203,
           1274, 128009]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1]], device='cuda:0')}

In [37]:
tokenizer.decode(encoded_inputs['input_ids'][0])

'<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nHow do clouds form?<|eot_id|><|start_header_id|>system<|end_header_id|>\n\nYou are an expert Meteorologist who can explain concept to lay people<|eot_id|>'

Let's check that we are using the quantized model:

In [33]:
model.get_memory_footprint()/(1024 * 1024 * 1024)

5.2075276374816895

To obtain text, we decode the generated output:

In [28]:
tokenizer.decode(outputs[0])

"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nHow do clouds form?<|eot_id|><|start_header_id|>system<|end_header_id|>\n\nYou are an expert Meteorologist who can explain concept to lay people<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nClouds are a fascinating phenomenon that play a crucial role in our planet's weather and climate. So, let me break down the process of how clouds form in a way that's easy to understand.\n\n**Step 1: Evaporation**\nClouds start forming when the sun heats the Earth's surface, causing water to evaporate from oceans, lakes, rivers, and even pudd"

Let's request for a longer answer. This time we do not set the limit and add and token to stop the genreation.

For the apply template:

- we separate the tokenization and formatting of the input.

First we format the input with the chat template adding a generation token prompt. This is is a string object.

Second, we use the tokenizer on the formatted_prompt to return a pyTorch tensor with the token IDs

In [29]:
# Tokenizer

tokenizer = AutoTokenizer.from_pretrained(base_model_insruct)
tokenizer.pad_token = tokenizer.eos_token

formatted_prompt = tokenizer.apply_chat_template(messages,
                                                 tokenize=False,
                                                 add_generation_prompt=True)

#inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
inputs = tokenizer(formatted_prompt, return_tensors="pt", padding=True).to("cuda")
attention_mask = inputs["attention_mask"]
outputs = model.generate(
  inputs['input_ids'],
  attention_mask=attention_mask,
  pad_token_id=tokenizer.eos_token_id
)

Note that we also asked to add_generation_prompt. This means adding the "assistant" token at the end. This prompts the model to generate an assistant reply automatically. This is different from the formatting above. Without the "assistant" token the model might continue the user message and not understand that it needs to behave as an assisant and generate a response.

In [30]:
formatted_prompt

'<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nHow do clouds form?<|eot_id|><|start_header_id|>system<|end_header_id|>\n\nYou are an expert Meteorologist who can explain concept to lay people<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'

In [31]:
tokenizer.decode(outputs[0])

"<|begin_of_text|><|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nHow do clouds form?<|eot_id|><|start_header_id|>system<|end_header_id|>\n\nYou are an expert Meteorologist who can explain concept to lay people<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nClouds! Those majestic, wispy wonders that bring shade, rain, and a touch of magic to our daily lives. So, let me break down the process of how clouds form in a way that's easy to understand.\n\n**The Basics**\n\nClouds are made up of tiny water droplets or ice crystals suspended in the air. They form when water vapor in the air condenses onto tiny particles, like dust, salt, or pollen. This process is called condensation.\n\n**The Process**\n\nHere's a step-by-step guide to cloud formation:\n\n1. **Evaporation**: The sun heats the Earth's surface, causing water to evaporate from oceans, lakes, rivers, and even your backyard pool! This water vapor rises into the air as gas.\n2. **Cooling**: As the water vapor

The best approach is to format and then tokenize as just shown above. Also add a generation prompt to avoid bug or make sure we are in a response assistant mode. In general you should use:

```
model.generate(
    input_ids=...,
    attention_mask=...,
    pad_token_id=tokenizer.eos_token_id
)
```

To avoid increased memory usage and attending to padded token during generation set the attention mask correctly as well as the pad token id. This increase increase in memory may be very important when using huggingFace.

In [47]:
del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()

#5.Conclusions

In this notebook, I showed how to use two open LLMs with HuggingFace. We examined the tokenizer for LLama 3 and Phi4 models and loaded models in memory. Since the models are large and require a lot of compute resource we use a T4 GPU with 15GB of memory from google colab.

Llama 3 8B model shows a size of nearly 15 GB size in the memory which

THere were three main goals in this exercise:

- familiarize with tokenizer
- understanding LLM model size
- introducing quantization
- exploring typical LLM model architecture
- illustrating how to generate text/do inference with LLM.



#6.References


Obregón, M.A., Rodrigues, G., Costa, M.J., et al. (2019). Validation of ESA Sentinel-2 L2A aerosol optical thickness and columnar water vapour during 2017-2018. Remote Sensing, 11(14), 1649. https://doi.org/10.3390/rs11141649

Schläpfer, D., Borel, C.C., Keller, J., et al. (1998). Atmospheric precorrected differential absorption technique to retrieve columnar water vapor. Remote Sensing of Environment, 65(3), 353-366. https://doi.org/10.1016/S0034-4257(98)00044-3.

Gascon F., Bouzinac C., Thépaut O., et al. (2017). Copernicus Sentinel-2A calibration and products validation status. Remote Sensing, 9(6), 584. https://doi.org/10.3390/rs9060584


In [ ]:
############################# END OF SCRIPT ###################################